# Module 5, Part 2: Advanced Agents — Practical Experiments

**Days 64–76 | Units 5.7–5.15**  
Companion: [detailed guide](Module-5-Part-2-Advanced-Agents-Detailed.md). Run cells in order. Fictional data only. Offline examples are simulations; optional paid and integration exercises are explicitly marked.

In [ ]:
import json, time
from pathlib import Path
POLICIES = {
    "leave": "[leave] Request planned leave 10 calendar days ahead; manager approval is required.",
    "expenses": "[expenses] Submit receipts within 30 days. Meals are capped at 20 fictional credits per day.",
    "access": "[access] Report a lost MFA device to the IT support team immediately."
}
def search_policy(query):
    words = set(query.lower().split())
    return [text for key, text in POLICIES.items() if key in words or words.intersection(text.lower().split())][:2]
print("Synthetic policy records:", len(POLICIES))

## 5.7–5.8: MCP design exercise

An MCP host connects through a client to a server. Design `search_policy(query)` and `list_topics()` as tools and `policy://about` as a resource. Read the linked MCP server guide, build a local server and inspect it with a client; do not expose real files or credentials. The following is only a capability manifest, not a running MCP server.

In [ ]:
manifest = {"tools": ["search_policy", "list_topics"], "resources": ["policy://about"]}
assert len(set(manifest["tools"])) == 2
print(manifest)

## 5.9: Memory scope, update and deletion

This dictionary is a teaching fixture, not durable storage or authorization. Server-side authenticated identity must determine the user key.

In [ ]:
memory = {}
def remember(user, key, value): memory[(user, key)] = value
def recall(user, key): return memory.get((user, key))
def forget(user):
    for key in list(memory):
        if key[0] == user: del memory[key]
remember("alice", "language", "English")
remember("alice", "language", "Hindi")
assert recall("alice", "language") == "Hindi" and recall("bob", "language") is None
forget("alice")
assert recall("alice", "language") is None
print("Memory fixture checks passed")

## 5.10–5.12: Context ledger and compaction

Character count is a rough size proxy, not a token count. Check exact facts after compression and evaluate a real summarizer separately.

In [ ]:
state = {"goal":"Resolve leave question", "constraints":["cite policy"], "facts":{"request_id":"NX-7421"}, "pending":["retrieve policy"]}
ledger = {key: len(json.dumps(value)) for key, value in state.items()}
summary = f"Goal: {state['goal']}; pending: {', '.join(state['pending'])}"
compacted = {"summary": summary, "constraints":state["constraints"], "facts":state["facts"]}
assert compacted["facts"]["request_id"] == "NX-7421"
print("Character ledger:", ledger, "Compacted:", compacted)

## 5.13–5.15: Supervisor and failure handling

Two deterministic specialists demonstrate routing and a failure; they do not represent actual model agents. Compare with a direct search on the same inputs.

In [ ]:
def supervisor(topic, fail=False):
    if topic == "policy":
        if fail: return {"status":"incomplete", "reason":"policy specialist unavailable", "calls":1}
        return {"status":"complete", "result": search_policy("leave"), "calls":1}
    if topic == "order": return {"status":"incomplete", "reason":"no order system configured", "calls":1}
    return {"status":"needs_clarification", "calls":0}
assert supervisor("policy")["result"]
assert supervisor("policy", fail=True)["status"] == "incomplete"
assert supervisor("unknown")["status"] == "needs_clarification"
print(supervisor("policy"), supervisor("policy", True))

### Experiment

Implement an actual local MCP server using the official guide, with two synthetic tools and one resource. Inspect the tool schemas in a client. Record package versions, transport and results; compare a single agent and a specialist design on ten tasks when real model access is available.

# Unit-by-unit practice

Run these cells in order after the introductory examples. All outcomes below are deterministic, fictional simulations unless a cell explicitly says otherwise. They do not establish real model performance, production security, persistent storage, or external integrations.

## 5.7 — Host, client and server boundaries

Represent capabilities and check a host allowlist. This is a protocol-shaped fixture, not an MCP connection.

In [ ]:
server_manifest = {"server":"northstar-demo", "tools":["search_policy", "list_topics"], "resources":["policy://about"]}
def expose_tools(manifest, allowlist): return sorted(set(manifest["tools"]) & set(allowlist))
assert expose_tools(server_manifest, {"search_policy"}) == ["search_policy"]
assert "delete_files" not in expose_tools(server_manifest, {"search_policy", "delete_files"})
print("Host exposes:", expose_tools(server_manifest, {"search_policy"}))

**Explore:** Draw the host → client → server → data path. Which process holds the credentials and which enforces permissions?

## 5.8 — Test a synthetic server contract

Implement the two read-only operations and resource as ordinary Python functions. They are not registered or transported through MCP.

In [ ]:
def demo_server_call(name, args):
    if name == "list_topics" and args == {}: return {"status":"ok", "topics":sorted(POLICIES)}
    if name == "search_policy" and set(args) == {"query"} and isinstance(args["query"], str) and args["query"].strip():
        return {"status":"ok", "results":search_policy(args["query"])[:2]}
    return {"status":"invalid", "error":"Unsupported operation or arguments"}
def demo_resource(uri):
    return "Synthetic Northstar policy corpus; no customer data." if uri == "policy://about" else None
assert demo_server_call("list_topics", {})["topics"] == sorted(POLICIES)
assert demo_server_call("search_policy", {"query":"leave"})["results"]
assert demo_server_call("search_policy", {"query":4})["status"] == "invalid"
assert demo_resource("policy://about") and demo_resource("policy://private") is None
print("Synthetic server contract passed")

**Explore:** Use the linked official MCP guide to expose these operations on a local server and connect a client. Record package version, transport and observed results; this fixture does not test MCP.

## 5.9 — Versioned memory and user isolation

This extends the earlier dictionary with source and revision. Identity is still a supplied test fixture, not authentication.

In [ ]:
records = {}
def put_memory(user, key, value, source):
    old = records.get((user, key), {})
    records[(user, key)] = {"value":value, "source":source, "revision":old.get("revision", 0)+1}
def get_memory(user, key): return records.get((user, key))
def delete_memory(user):
    for k in list(records):
        if k[0] == user: del records[k]
put_memory("alice", "language", "English", "user")
put_memory("alice", "language", "Hindi", "user correction")
put_memory("bob", "language", "French", "user")
assert get_memory("alice", "language")["revision"] == 2
assert get_memory("alice", "language")["value"] == "Hindi"
delete_memory("alice")
assert get_memory("alice", "language") is None and get_memory("bob", "language")["value"] == "French"
print("Memory correction and isolation fixture passed")

**Explore:** How would you enforce owner scope on a real database query and verify deletion of derived indexes, logs and backups?

## 5.10 — Build and inspect a context ledger

Character counts illustrate relative size only; they are not token counts or a cost estimate.

In [ ]:
context = {"instructions":"Cite evidence; do not write without approval.",
           "question":"When should I request leave?", "history":"User asks about leave.",
           "evidence":POLICIES["leave"], "tool_schema":"search_policy(query: string)"}
ledger = [{"component":k, "characters":len(v)} for k,v in context.items()]
assert sum(row["characters"] for row in ledger) == sum(map(len, context.values()))
for row in ledger: print(row)

**Explore:** Compare a ledger after 1, 10 and 20 steps of your own trace. Which items should remain exact, be summarized, or be retrieved again?

## 5.11 — Validate compaction preserves required facts

The summary is handwritten to make expected behavior deterministic. A real model-generated summary requires independent checks.

In [ ]:
def check_compaction(original, compacted):
    required = ("goal", "constraints", "facts", "pending")
    return [key for key in required if compacted.get(key) != original.get(key)]
original = {"goal":"Resolve leave question", "constraints":["approval required"], "facts":{"id":"NX-7421"}, "pending":["search"], "history":["greeting"]*20}
good = {k:original[k] for k in ("goal", "constraints", "facts", "pending")}
bad = {**good, "facts":{"id":"NX-742"}}
assert check_compaction(original, good) == []
assert check_compaction(original, bad) == ["facts"]
print("Preserved:", good, "Lost in faulty summary:", check_compaction(original, bad))

**Explore:** Compare the size of original and compacted state, then explain why character reduction alone is not proof of a better agent.

## 5.12 — Compare harness variants on a matched dataset

Toy outcomes demonstrate how to compare changes; these are not measured model results.

In [ ]:
harness_cases = [{"id":"a", "baseline":True, "candidate":True}, {"id":"b", "baseline":False, "candidate":True}, {"id":"c", "baseline":True, "candidate":False}]
def compare_harness(rows):
    return {"baseline":sum(r["baseline"] for r in rows), "candidate":sum(r["candidate"] for r in rows),
            "regressions":[r["id"] for r in rows if r["baseline"] and not r["candidate"]], "total":len(rows)}
comparison = compare_harness(harness_cases)
assert comparison == {"baseline":2, "candidate":2, "regressions":["c"], "total":3}
print(comparison)

**Explore:** The aggregate is unchanged but one task regressed. What extra information would you need before adopting the candidate?

## 5.13 — Design specialist handoffs

Validate required handoff fields before dispatching. This does not make the receiving specialist trustworthy.

In [ ]:
def valid_handoff(task):
    return all(task.get(k) for k in ("goal", "scope", "deadline", "required_output"))
handoff = {"goal":"Find leave policy", "scope":"public-policy", "deadline":"30s", "required_output":"finding and source IDs"}
assert valid_handoff(handoff)
assert not valid_handoff({k:v for k,v in handoff.items() if k != "scope"})
print("Handoff:", handoff)

**Explore:** Identify what information should be omitted from this specialist and what happens if its output conflicts with another result.

## 5.14 — Merge specialist results without inventing success

A supervisor can return a partial result when a required worker fails. Workers and results here are synthetic.

In [ ]:
def merge_specialists(results, required):
    missing = [name for name in required if name not in results or results[name].get("status") != "ok"]
    return {"status":"partial" if missing else "complete", "missing":missing,
            "findings":{k:v["finding"] for k,v in results.items() if v.get("status") == "ok"}}
merged = merge_specialists({"policy":{"status":"ok", "finding":"10 days [leave]"}, "order":{"status":"unavailable"}}, ["policy", "order"])
assert merged["status"] == "partial" and merged["missing"] == ["order"]
assert "order" not in merged["findings"]
print(merged)

**Explore:** Add a conflict case where two sources disagree. Return the conflicting evidence rather than silently selecting one.

## 5.15 — Cancellation and independent budgets

The function models a decision point before an action. It does not interrupt already-running remote requests.

In [ ]:
def may_start_action(state):
    if state["cancelled"]: return False, "cancelled"
    if state["calls"] >= state["call_limit"]: return False, "call_limit"
    if state["elapsed"] >= state["time_limit"]: return False, "time_limit"
    return True, "allowed"
assert may_start_action({"cancelled":True, "calls":0,"call_limit":2,"elapsed":0,"time_limit":10}) == (False, "cancelled")
assert may_start_action({"cancelled":False, "calls":2,"call_limit":2,"elapsed":0,"time_limit":10}) == (False, "call_limit")
assert may_start_action({"cancelled":False, "calls":0,"call_limit":2,"elapsed":10,"time_limit":10}) == (False, "time_limit")
print("Cancellation and budget checks passed")

**Explore:** How can the application explain an in-flight external action whose outcome is unknown after cancellation?

## Capstone and reflection

Complete this part's capstone in the companion guide. Record expected behavior before running, actual evidence, failures, limitations and improvements. Keep deterministic results separate from any live model or integration results.